# 05 - Run All and Launch Streamlit
يشغّل الـpipeline المحلي لكل كاميرا، ثم يفتح Streamlit اختياريًا على http://localhost:8501.

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'Data' / 'raw'
APP_PATH = PROJECT_ROOT / 'Output' / 'app' / 'streamlit_app.py'
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv'}
RUN_PIPELINE_IF_VIDEOS = True
START_STREAMLIT = False  # Set to True only when you want to launch the app after the pipeline.

videos = sorted(path for path in RAW_DIR.iterdir() if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS)
print(f'Found {len(videos)} direct video file(s) in Data/raw.')
print('Nested folders are intentionally excluded so every selected camera has an explicit zone calibration.')


In [ ]:
if RUN_PIPELINE_IF_VIDEOS and videos:
    import nbformat
    from nbclient import NotebookClient

    pipeline = [
        '00_Project_Setup.ipynb',
        '01_Local_Detection_Tracking.ipynb',
        '02_Global_Fusion_and_Zones.ipynb',
        '03_Retail_Analytics_Agent.ipynb',
        '04_Streamlit_Dashboard.ipynb',
    ]
    for notebook_name in pipeline:
        notebook_path = PROJECT_ROOT / 'Notebook' / notebook_name
        print(f'Running {notebook_name} ...')
        notebook = nbformat.read(notebook_path, as_version=4)
        client = NotebookClient(
            notebook, timeout=None, kernel_name='python3',
            resources={'metadata': {'path': str(PROJECT_ROOT)}}
        )
        client.execute()
        nbformat.write(notebook, notebook_path)
    print('Camera-local pipeline completed.')
elif not videos:
    print('No videos found. Add videos to Data/raw before running the pipeline.')


In [ ]:
if START_STREAMLIT:
    command = [sys.executable, '-m', 'streamlit', 'run', str(APP_PATH), '--server.port', '8501']
    process = subprocess.Popen(command, cwd=PROJECT_ROOT)
    print(f'Streamlit started with PID {process.pid}. Open http://localhost:8501')
    print('Stop it later by interrupting its terminal process.')
